In [14]:
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from torch import nn

In [15]:
df = pd.read_csv("test.csv")
df.head()
X, y = df["X"], df["y"]

X = torch.from_numpy(X.to_numpy()).type(dtype=torch.float32)
y = torch.from_numpy(y.to_numpy()).type(dtype=torch.float32)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [16]:
class Regularization_Term(nn.Module):
    def __init__(self):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(1))
        self.bias = nn.Parameter(torch.rand(1))
    def forward(self,X):
        return self.weight*X+self.bias

In [17]:
def Acc_Func(y_true, y_pred):
    return (torch.eq(y_true, y_pred).sum().item() / len(y_true)) * 100

In [18]:
model30 = Regularization_Term()
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(params=model30.parameters(), lr=0.001, weight_decay=1)
epochs = 200

lambda_L2 = 0.01
lambda_L1 = 0.01
for epoch in range(epochs):
    model30.train()
    y_preds = model30(X_train)
    loss = (
        loss_fn(y_preds, y_train)
        + lambda_L1 * torch.sum(torch.abs(model30.weight))
        + lambda_L2 * torch.sum(model30.weight**2)
    )
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    model30.eval()
    with torch.inference_mode():
        test_preds = model30(X_test)
        test_loss = loss_fn(test_preds, y_test)

        # Round predictions to nearest integer
        y_pred_round = torch.round(test_preds)

        acc = Acc_Func(y_test, y_pred_round)
        if epoch % 20 == 0:
            print(
                f"Epoch: {epoch}, Loss: {loss:.6f}, Test Loss: {test_loss:.6f}, Accuracy: {acc:.2f}%"
            )

Epoch: 0, Loss: 134.426178, Test Loss: 40.752857, Accuracy: 0.00%
Epoch: 20, Loss: 0.276817, Test Loss: 0.296110, Accuracy: 66.67%
Epoch: 40, Loss: 0.209586, Test Loss: 0.224741, Accuracy: 66.67%
Epoch: 60, Loss: 0.206868, Test Loss: 0.220648, Accuracy: 66.67%
Epoch: 80, Loss: 0.204759, Test Loss: 0.217543, Accuracy: 66.67%
Epoch: 100, Loss: 0.202733, Test Loss: 0.214560, Accuracy: 66.67%
Epoch: 120, Loss: 0.200779, Test Loss: 0.211679, Accuracy: 66.67%
Epoch: 140, Loss: 0.198894, Test Loss: 0.208899, Accuracy: 66.67%
Epoch: 160, Loss: 0.197074, Test Loss: 0.206215, Accuracy: 66.67%
Epoch: 180, Loss: 0.195318, Test Loss: 0.203622, Accuracy: 66.67%


In [19]:
print(model30.weight.item())
print(model30.bias.item())

2.0574862957000732
0.25177934765815735


In [20]:
from sklearn.metrics import r2_score
r2 = r2_score(y_test.numpy(),test_preds.numpy())
print(f"R2 Score: {r2:.4f}")

R2 Score: 0.9927


In [21]:
for name, param in model30.named_parameters():
    if param.grad is not None:
        print(name, param.grad)

weight tensor([-2.0338])
bias tensor([-0.5097])


In [22]:
from sklearn.linear_model import Ridge

ridge_reg = Ridge(alpha=1, solver="cholesky")
ridge_reg.fit(X.reshape(-1, 1), y)

Ridge(alpha=1, solver='cholesky')

In [23]:
print(ridge_reg.predict([[2]]))
print(model30.weight.item()*2+model30.bias.item())

[5.04270463]
4.366751939058304


In [ ]:
from sklearn.linear_model import  ElasticNet
elastic_net = ElasticNet(alpha=1,l1_ratio=0.5)
elastic_net.fit(X.reshape(-1,1),y)



ElasticNet(alpha=1)

In [27]:
elastic_net.predict(X=[[2]])

array([5.46956522])